# Lab 01: ER-to-Relational DDL & Integrity Constraints

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

SQLite schema creation with PRIMARY KEY, FOREIGN KEY ON DELETE CASCADE, CHECK, and UNIQUE constraints.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB01_ER_DDL_and_Constraints.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. ER-to-Relational DDL with Primary, Foreign, Check, and Unique Constraints

Mapping an M:N relationship (`Student` enrolls in `Course`) produces a junction table `Enrolment(SID, CID, Grade)` with a composite primary key `(SID, CID)` and cascading foreign keys.

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript("""
CREATE TABLE Student (
    SID   INTEGER PRIMARY KEY AUTOINCREMENT,
    Name  TEXT NOT NULL,
    Dept  TEXT NOT NULL,
    GPA   REAL CHECK (GPA IS NULL OR (GPA BETWEEN 0.00 AND 4.00)),
    Email TEXT UNIQUE
);
CREATE TABLE Course (
    CID     TEXT PRIMARY KEY,
    CName   TEXT NOT NULL,
    Credits INTEGER NOT NULL DEFAULT 3 CHECK (Credits IN (1, 2, 3, 4))
);
CREATE TABLE Enrolment (
    SID   INTEGER NOT NULL,
    CID   TEXT NOT NULL,
    Grade TEXT CHECK (Grade IN ('A','B','C','D','F') OR Grade IS NULL),
    PRIMARY KEY (SID, CID),
    FOREIGN KEY (SID) REFERENCES Student(SID) ON DELETE CASCADE,
    FOREIGN KEY (CID) REFERENCES Course(CID) ON DELETE CASCADE
);
""")
conn.executemany("INSERT INTO Student (Name, Dept, GPA, Email) VALUES (?, ?, ?, ?)", [
    ("Asha Ali", "CSE", 3.80, "asha@iitmz.example"),
    ("Omar Said", "CSE", 3.20, "omar@iitmz.example"),
])
conn.executemany("INSERT INTO Course VALUES (?, ?, ?)", [("DB101", "Databases", 3), ("CS102", "Data Structures", 4)])
conn.executemany("INSERT INTO Enrolment VALUES (?, ?, ?)", [(1, "DB101", "A"), (1, "CS102", "A"), (2, "DB101", "B")])
print("Students:", conn.execute("SELECT SID, Name, Dept, GPA FROM Student").fetchall())


Students: [(1, 'Asha Ali', 'CSE', 3.8), (2, 'Omar Said', 'CSE', 3.2)]


## 2. Testing Integrity Enforcement & `ON DELETE CASCADE`

Constraint failures raise `sqlite3.IntegrityError`, and deleting parent `Student(SID=1)` automatically cascades to `Enrolment`.

In [2]:
for bad_sql in [
    "INSERT INTO Student (Name, Dept, GPA, Email) VALUES ('Bad', 'CSE', 4.50, 'b@x.tz')",
    "INSERT INTO Enrolment VALUES (99, 'DB101', 'A')",
]:
    try:
        conn.execute(bad_sql)
        raise AssertionError("Constraint should have failed")
    except sqlite3.IntegrityError as exc:
        print("Caught expected IntegrityError:", exc)

conn.execute("DELETE FROM Student WHERE SID = 1")
left = conn.execute("SELECT * FROM Enrolment").fetchall()
print("Enrolment rows after deleting SID=1:", left)
assert left == [(2, "DB101", "B")]


Caught expected IntegrityError: CHECK constraint failed: GPA IS NULL OR (GPA BETWEEN 0.00 AND 4.00)
Caught expected IntegrityError: FOREIGN KEY constraint failed
Enrolment rows after deleting SID=1: [(2, 'DB101', 'B')]
